In [8]:
import numpy as np
import pandas as pd
import os
import time
from tqdm import tqdm, trange
import sys
import matplotlib.pyplot as plt
import pickle
import random
import math

# variaveis
L = 100 # lado do lattice
n_lagartos = L**2 # lagartos que cabem no lattice
estrategias = ['O', 'Y', 'B'] # estratégias possíveis
index_map = {'O': 0, 'Y': 1, 'B': 2}
n_geracoes = 200
n_pop = 50 # número de populações independentes
prob_mutacao = None # probabilidade de mutação a cada geração
tipo = f"teste\\learning_diferente"
vizinho_aprendizado_Y = 4

output_dir = "C:\\Unicamp\\mestrado\\simulacoes\\RPS-python\\RPS-POO\\outputs\\vizinhanca_aprendizado-interacao\\" + tipo + f"/vizinho_aprendizado_Y_{str(vizinho_aprendizado_Y)}/"
#os.makedirs(output_dir, exist_ok=True)

In [29]:
class Lagarto:
  def __init__(self, i, j, estrategia, fitness, coord_vizinhos_interacao, estrategia_vizinhos_interacao, coord_vizinhos_aprendizado, estrategia_vizinhos_aprendizado, t, n_vizinhos_interacao, n_vizinhos_aprendizado):
    self.i = i # linha
    self.j = j # coluna
    self.estrategia = estrategia
    self.fitness = 0 # inicia com 0 de fitness
    self.coord_vizinhos_interacao = [] # lista vazia para adicionar as coordenadas dos vizinhos
    self.estrategia_vizinhos_interacao = [] # lista vazia para adicionar as estratégias dos vizinhos
    self.coord_vizinhos_aprendizado = []
    self.estrategia_vizinhos_aprendizado = []
    self.t = t  # determina a geracao do lagarto
    self.n_vizinhos_interacao = n_vizinhos_interacao # número de vizinhos que o lagarto efetivamente joga
    self.n_vizinhos_aprendizado = n_vizinhos_aprendizado # número de vizinhos que o lagarto olha para aprender

  def calcular_coord_vizinhos_custom(self, L, tipo):
    if tipo == 'interacao':
        # Calcula o menor raio necessário para abranger pelo menos n_vizinhos
        n_vizinhos = self.n_vizinhos_interacao
        raio = math.ceil((math.sqrt(n_vizinhos + 1) - 1) / 2)
        vizinhos = []
        for dx in range(-raio, raio + 1):
            for dy in range(-raio, raio + 1):
                if dx == 0 and dy == 0:
                    continue
                ni = (self.i + dx) % L
                nj = (self.j + dy) % L
                vizinhos.append((ni, nj))
        vizinhos = list(set(vizinhos))  # remove duplicatas (em grid toroidal pode não ser necessário)
        # Sorteia n_vizinhos sem repetição
        if len(vizinhos) < n_vizinhos:
            raise ValueError(f"Não há vizinhos suficientes: pedido={n_vizinhos}, disponíveis={len(vizinhos)}")
        selecionados = random.sample(vizinhos, n_vizinhos)
        self.coord_vizinhos_interacao = selecionados

    elif tipo == 'aprendizado':
        n_vizinhos = self.n_vizinhos_aprendizado
        raio = math.ceil((math.sqrt(n_vizinhos + 1) - 1) / 2)
        vizinhos = []
        for dx in range(-raio, raio + 1):
            for dy in range(-raio, raio + 1):
                if dx == 0 and dy == 0:
                    continue
                ni = (self.i + dx) % L
                nj = (self.j + dy) % L
                vizinhos.append((ni, nj))
        vizinhos = list(set(vizinhos))  # remove duplicatas (em grid toroidal pode não ser necessário)
        # Sorteia n_vizinhos sem repetição
        if len(vizinhos) < n_vizinhos:
            raise ValueError(f"Não há vizinhos suficientes: pedido={n_vizinhos}, disponíveis={len(vizinhos)}")
        selecionados = random.sample(vizinhos, n_vizinhos)
        self.coord_vizinhos_aprendizado = selecionados
    
  def obter_estrategia_vizinhos(self, matriz_posicao):
      self.estrategia_vizinhos_interacao = [matriz_posicao[ni, nj] for ni, nj in self.coord_vizinhos_interacao] # dadas as coordenadas, obtém a estratégia do lagarto que ocupa aquela posição
      self.estrategia_vizinhos_aprendizado = [matriz_posicao[ni, nj] for ni, nj in self.coord_vizinhos_aprendizado] # dadas as coordenadas, obtém a estratégia do lagarto que ocupa aquela posição

  def mutacao(self, prob_mutacao): # função de mutação
    if np.random.rand() < prob_mutacao: # sorteia um valor entre 0 e 1, se for menor que a probabilidade de mutação, o lagarto muda de estratégia
        estrategias_possiveis = [e for e in estrategias if e != self.estrategia] # obtém as estratégias possíveis, exceto a atual
        self.estrategia = np.random.choice(estrategias_possiveis) # escolhe uma nova estratégia aleatoriamente para mutar

  def adicionar_vizinhos_inicial(self, vizinho_aprendizado_Y):
      if self.estrategia == 'Y':
          #n_vizinhos_interacao = np.random.randint(1, 9)
          n_vizinhos_interacao = 24
          self.n_vizinhos_interacao = n_vizinhos_interacao
          #n_vizinhos_aprendizado = np.random.randint(1, 9)
          n_vizinhos_aprendizado = vizinho_aprendizado_Y
          self.n_vizinhos_aprendizado = n_vizinhos_aprendizado
      elif self.estrategia == 'O':
          #n_vizinhos_interacao = np.random.randint(1, 9)
          n_vizinhos_interacao = 24
          self.n_vizinhos_interacao = n_vizinhos_interacao
          #n_vizinhos_aprendizado = np.random.randint(1, 9)
          n_vizinhos_aprendizado = 24
          self.n_vizinhos_aprendizado = n_vizinhos_aprendizado
      elif self.estrategia == 'B':
          #n_vizinhos_interacao = np.random.randint(1, 9)
          n_vizinhos_interacao = 24
          self.n_vizinhos_interacao = n_vizinhos_interacao
          #n_vizinhos_aprendizado = np.random.randint(1, 9)
          n_vizinhos_aprendizado = 24
          self.n_vizinhos_aprendizado = n_vizinhos_aprendizado
###

def calcular_media_vizinhos(lagartos, estrategias, tipo):
    if tipo == 'interacao':
        medias_interacao = []
        for e in estrategias:
            #viz = [lag.n_vizinhos_realizado for lag in lagartos if lag.estrategia == e]
            viz = [lag.n_vizinhos_interacao for lag in lagartos if lag.estrategia == e]
            medias_interacao.append(np.mean(viz) if len(viz) > 0 else 0)
        return medias_interacao # retorna a média de vizinhos para cada estratégia

    elif tipo == 'aprendizado':
        medias_aprendizado = []
        for e in estrategias:
            viz = [lag.n_vizinhos_aprendizado for lag in lagartos if lag.estrategia == e]
            medias_aprendizado.append(np.mean(viz) if len(viz) > 0 else 0)
        return medias_aprendizado # retorna a média de vizinhos para cada estratégia

In [30]:
# Teste função calcular_coord_vizinhos_custom()

# --- parâmetros mínimos ---
L = 7

# --- matriz de estratégias ---
np.random.seed(0)
estrategias = ['O', 'Y', 'B']
matriz_posicao = np.random.choice(estrategias, size=(L, L))

print("Matriz de estratégias:")
print(matriz_posicao)

lag = Lagarto(i=3,
              j=3,
              estrategia='Y',
              fitness=0,
              coord_vizinhos_interacao=[],
              estrategia_vizinhos_interacao=[],
              coord_vizinhos_aprendizado=[],
              estrategia_vizinhos_aprendizado=[],
              t=0,
              n_vizinhos_interacao=24,
              n_vizinhos_aprendizado=4)

# --- chama a função que queremos testar ---
lag.calcular_coord_vizinhos_custom(L=L, tipo='aprendizado')
lag.calcular_coord_vizinhos_custom(L=L, tipo='interacao')
lag.obter_estrategia_vizinhos(matriz_posicao=matriz_posicao)

# --- imprime o resultado ---
print("\nCoordenadas dos vizinhos aprendizado:")
print(lag.coord_vizinhos_aprendizado)
print(f"\nTotal de vizinhos aprendizado: {len(lag.coord_vizinhos_aprendizado)}")

print("\nEstratégias dos vizinhos aprendizado:")
print(lag.estrategia_vizinhos_aprendizado)


print("\nCoordenadas dos vizinhos interação:")
print(lag.coord_vizinhos_interacao)
print(f"\nTotal de vizinhos interação: {len(lag.coord_vizinhos_interacao)}")

print("\nEstratégias dos vizinhos interação:")
print(lag.estrategia_vizinhos_interacao)

Matriz de estratégias:
[['O' 'Y' 'O' 'Y' 'Y' 'B' 'O']
 ['B' 'O' 'O' 'O' 'B' 'Y' 'B']
 ['B' 'O' 'Y' 'Y' 'Y' 'Y' 'O']
 ['Y' 'O' 'O' 'Y' 'B' 'O' 'B']
 ['O' 'Y' 'Y' 'B' 'O' 'Y' 'Y']
 ['Y' 'O' 'B' 'O' 'B' 'B' 'O']
 ['B' 'O' 'O' 'O' 'Y' 'Y' 'B']]

Coordenadas dos vizinhos aprendizado:
[(4, 2), (2, 3), (2, 2), (4, 3)]

Total de vizinhos aprendizado: 4

Estratégias dos vizinhos aprendizado:
['Y', 'Y', 'Y', 'B']

Coordenadas dos vizinhos interação:
[(2, 3), (1, 2), (1, 5), (3, 1), (5, 4), (3, 4), (2, 4), (2, 5), (4, 2), (5, 3), (2, 1), (5, 1), (1, 4), (3, 5), (5, 5), (3, 2), (4, 5), (1, 1), (2, 2), (5, 2), (1, 3), (4, 4), (4, 3), (4, 1)]

Total de vizinhos interação: 24

Estratégias dos vizinhos interação:
['Y', 'O', 'Y', 'O', 'B', 'B', 'Y', 'Y', 'Y', 'O', 'O', 'O', 'B', 'O', 'B', 'O', 'Y', 'O', 'Y', 'B', 'O', 'O', 'B', 'Y']


In [26]:
# Teste função adicionar_vizinhos_inicial()

# --- cria alguns lagartos para testar ---
lagartos_teste = [
    Lagarto(i=0, j=0, estrategia='Y', fitness=0,
            coord_vizinhos_interacao=[], estrategia_vizinhos_interacao=[],
            coord_vizinhos_aprendizado=[], estrategia_vizinhos_aprendizado=[],
            t=0, n_vizinhos_interacao=0, n_vizinhos_aprendizado=0),
    Lagarto(i=1, j=1, estrategia='O', fitness=0,
            coord_vizinhos_interacao=[], estrategia_vizinhos_interacao=[],
            coord_vizinhos_aprendizado=[], estrategia_vizinhos_aprendizado=[],
            t=0, n_vizinhos_interacao=0, n_vizinhos_aprendizado=0),
    Lagarto(i=2, j=2, estrategia='B', fitness=0,
            coord_vizinhos_interacao=[], estrategia_vizinhos_interacao=[],
            coord_vizinhos_aprendizado=[], estrategia_vizinhos_aprendizado=[],
            t=0, n_vizinhos_interacao=0, n_vizinhos_aprendizado=0),
    Lagarto(i=3, j=3, estrategia='Y', fitness=0,
            coord_vizinhos_interacao=[], estrategia_vizinhos_interacao=[],
            coord_vizinhos_aprendizado=[], estrategia_vizinhos_aprendizado=[],
            t=0, n_vizinhos_interacao=0, n_vizinhos_aprendizado=0),
    Lagarto(i=4, j=4, estrategia='O', fitness=0,
            coord_vizinhos_interacao=[], estrategia_vizinhos_interacao=[],
            coord_vizinhos_aprendizado=[], estrategia_vizinhos_aprendizado=[],
            t=0, n_vizinhos_interacao=0, n_vizinhos_aprendizado=0),
    Lagarto(i=5, j=5, estrategia='B', fitness=0,
            coord_vizinhos_interacao=[], estrategia_vizinhos_interacao=[],
            coord_vizinhos_aprendizado=[], estrategia_vizinhos_aprendizado=[],
            t=0, n_vizinhos_interacao=0, n_vizinhos_aprendizado=0),
    Lagarto(i=6, j=6, estrategia='Y', fitness=0,
            coord_vizinhos_interacao=[], estrategia_vizinhos_interacao=[],
            coord_vizinhos_aprendizado=[], estrategia_vizinhos_aprendizado=[],
            t=0, n_vizinhos_interacao=0, n_vizinhos_aprendizado=0),
]

# --- aplica a função ---
for lagarto in lagartos_teste:
    lagarto.adicionar_vizinhos_inicial(vizinho_aprendizado_Y = 40)

# --- imprime resultados ---
for lagarto in lagartos_teste:
    print(f"Estratégia: {lagarto.estrategia}")
    print(f"  n_vizinhos_interacao: {lagarto.n_vizinhos_interacao}")
    print(f"  n_vizinhos_aprendizado: {lagarto.n_vizinhos_aprendizado}")

Estratégia: Y
  n_vizinhos_interacao: 24
  n_vizinhos_aprendizado: 40
Estratégia: O
  n_vizinhos_interacao: 24
  n_vizinhos_aprendizado: 24
Estratégia: B
  n_vizinhos_interacao: 24
  n_vizinhos_aprendizado: 24
Estratégia: Y
  n_vizinhos_interacao: 24
  n_vizinhos_aprendizado: 40
Estratégia: O
  n_vizinhos_interacao: 24
  n_vizinhos_aprendizado: 24
Estratégia: B
  n_vizinhos_interacao: 24
  n_vizinhos_aprendizado: 24
Estratégia: Y
  n_vizinhos_interacao: 24
  n_vizinhos_aprendizado: 40


In [27]:
# Teste função calcular_media_vizinhos()

medias_interacao = calcular_media_vizinhos(lagartos_teste, estrategias, tipo='interacao')
medias_aprendizado = calcular_media_vizinhos(lagartos_teste, estrategias, tipo='aprendizado')

print("Médias de vizinhos (interação) por estratégia:")
for e, m in zip(estrategias, medias_interacao):
    print(f"  {e}: {m}")

print("\nMédias de vizinhos (aprendizado) por estratégia:")
for e, m in zip(estrategias, medias_aprendizado):
    print(f"  {e}: {m}")

Médias de vizinhos (interação) por estratégia:
  O: 24.0
  Y: 24.0
  B: 24.0

Médias de vizinhos (aprendizado) por estratégia:
  O: 24.0
  Y: 40.0
  B: 24.0
